# 12 — K-Means con voto di maggioranza

Un modello per notebook, con API originali sklearn e passaggi distinti.
KDDTrain+ completo, tutte le 41 feature, cinque famiglie. Cinque split 80/20 stratificati,
seed 1–5: gli stessi record dei modelli precedenti. Il test ufficiale resta separato.
Una configurazione iniziale fissata prima delle prove, senza ricerca di iperparametri.


## Come ragiona questo modello

`KMeans`, da `sklearn.cluster`, impara300 centroidi per rappresentare gruppi vicini
nello spazio delle feature trasformate. Non legge le etichette durante il fit.
I300 cluster non sono300classi: **dopo il fit**, ogni cluster riceve la famiglia
più frequente fra i suoi record di training. Più cluster possono eleggere la
stessa famiglia; una classe rara può non vincere in nessuno.

Il paper esplora300–400 cluster; fissiamo300, initk-means++, n_init10, Lloyd,
max_iter300 e tol0.0001. Ogni fit prova10 inizializzazioni e sceglie per inerzia
di training: cinque fit comportano50 ottimizzazioni interne, senza tuning su label.
Pareggi nel voto risolti con l'ordine Normal,DoS,Probe,R2L,U2R; eventuali cluster
vuoti ricevono la maggioranza globale del training. Lo spareggio deterministico
è un adattamento: il paper usa uno spareggio casuale.

In validation scegliamo il centro più vicino e applichiamo il mapping già fissato
sul training. **Non ri-etichettiamo i cluster guardando le risposte di validation.**
[KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER

## 1. Leggere i dati e distinguere input e target

`pd.read_csv` legge il CSV locale senza intestazione. Schema e mapping sono
costanti del progetto, non funzioni degli autori del dataset. Le 41 feature
escludono `attack_label` e `difficulty`; il target è la famiglia dell'attacco.

In [ ]:
train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
data = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
unknown = data['attack_label'].isna() | ~data['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette mancanti o non mappate: riesaminare il target.')

X = data.loc[:, list(FEATURE_COLUMNS)].copy()
y = data['attack_label'].map(ATTACK_TO_FAMILY).rename('family')
assert X.shape == (125973, 41) and X.index.equals(pd.RangeIndex(len(X)))
assert not X.isna().any().any()
display(X.head())
display(y.value_counts().reindex(FAMILY_ORDER).rename('record'))

## 2. Fissare i cinque split prima del training

`train_test_split(..., test_size=0.2, stratify=y, random_state=seed)` è la funzione
originale sklearn. Verifichiamo hash, protocollo, indici e target del riferimento
congelato dal05 prima di addestrare. Ogni run usa il proprio seed 1–5 anche per
l'inizializzazione del modello: la dispersione combina split e inizializzazione,
non separa le due cause. Nessun nuovo split viene scelto dopo i risultati.


In [ ]:
from sklearn.model_selection import train_test_split

reference_path = project_root / 'artifacts/rare_class_reference'
reference = json.loads((reference_path / 'results.json').read_text(encoding='utf-8'))
source_sha256 = hashlib.sha256(train_path.read_bytes()).hexdigest()
assert source_sha256 == reference['source_sha256']
assert reference['seeds'] == [1, 2, 3, 4, 5]
assert reference['validation_size'] == 0.2 and reference['feature_count'] == 41
assert reference['family_order'] == list(FAMILY_ORDER)
with np.load(reference_path / 'predictions.npz', allow_pickle=False) as saved:
    reference_arrays = {key: saved[key].copy() for key in saved.files}

seeds = [1, 2, 3, 4, 5]
prepared_splits = []
split_records = []
for run, seed in enumerate(seeds, start=1):
    X_train, X_validation, y_train, y_validation = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed,
    )
    train_indices = X_train.index.to_numpy()
    validation_indices = X_validation.index.to_numpy()
    assert not np.intersect1d(train_indices, validation_indices).size
    prepared_splits.append({
        'run': run, 'seed': seed, 'train_indices': train_indices,
        'validation_indices': validation_indices, 'y_validation': y_validation,
    })
    split_records.append({
        'run': run, 'seed': seed, 'train_rows': len(X_train), 'validation_rows': len(X_validation),
        'train_indices_sha256': hashlib.sha256(train_indices.tobytes()).hexdigest(),
        'validation_indices_sha256': hashlib.sha256(validation_indices.tobytes()).hexdigest(),
    })
assert split_records == reference['splits']
display(pd.DataFrame(split_records)[['run', 'seed', 'train_rows', 'validation_rows']])

In [ ]:
for split in prepared_splits:
    run = split['run']
    np.testing.assert_array_equal(
        split['validation_indices'], reference_arrays[f'run_{run:02d}_validation_indices'],
    )
    np.testing.assert_array_equal(
        np.asarray(split['y_validation']), reference_arrays[f'run_{run:02d}_truth'],
    )
print('Tutti i record e i target coincidono: possiamo procedere al training.')

## 3. Definire il preprocessing e la Pipeline

OneHotEncoder denso sulle tre categoriche; StandardScaler sulle38 numeriche.
Lo scaler rende confrontabili le unità numeriche; le dummy restano0/1.
La Pipeline apprende vocabolario e statistiche solo dal training di ciascuna run.
Le41 feature originali producono più colonne dopo il one-hot. Conserviamo anche
le costanti già documentate; nessuna nuova selezione o trasformazione delle label.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans

numeric_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
    ('numeric', StandardScaler(), numeric_columns),
], remainder='drop')
cluster_count = 300
classifier = KMeans(
    n_clusters=cluster_count, init='k-means++', n_init=10,
    max_iter=300, tol=1e-4, algorithm='lloyd',
)
model_template = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', classifier),
])
model_id = 'kmeans'
display(model_template)

## 4. Training: cinque fit, senza usare la validation

Cloniamo la Pipeline per run e impostiamo il seed del modello. Tempi, parametri
e tutti i warning vengono registrati. La predizione resta nel passaggio successivo.
K-Means riceve soltanto X_train: il target non partecipa all'ottimizzazione
dei centroidi. `n_init=10` esegue dieci inizializzazioni per chiamata fit.


In [ ]:
import warnings

fit_warnings_by_seed = {}
fitted_models = {}
fit_seconds_by_seed = {}
parameters_by_seed = {}
for split in prepared_splits:
    seed = split['seed']
    train_indices = split['train_indices']
    X_train = X.loc[train_indices]
    y_train = y.loc[train_indices]
    model = clone(model_template)
    model.set_params(classifier__random_state=seed)
    print(f"Inizio training {model_id}, seed={seed} ({len(X_train)} record).", flush=True)
    start = perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model.fit(X_train)
    fit_warnings_by_seed[str(seed)] = [
        {'category': w.category.__name__, 'message': str(w.message)} for w in caught
    ]
    for warning in fit_warnings_by_seed[str(seed)]:
        print(f"{warning['category']}: {warning['message']}", flush=True)
    fit_seconds_by_seed[str(seed)] = perf_counter() - start
    fitted_models[seed] = model
    parameters_by_seed[str(seed)] = model.named_steps['classifier'].get_params()
    print(f"Fit completato: {fit_seconds_by_seed[str(seed)]:.2f} secondi.", flush=True)

## 5. Controllare che il preprocessing abbia visto soltanto il training

Confrontiamo vocabolario e statistiche apprese con i record effettivi di training.
Registriamo anche eventuali categorie presenti soltanto in validation. Questi
controlli non cambiano dati, iperparametri o modello dopo aver osservato i risultati.

In [ ]:
preprocessing_checks = []
classifier_checks = []
for split in prepared_splits:
    seed = split['seed']
    X_train = X.loc[split['train_indices']]
    X_validation = X.loc[split['validation_indices']]
    y_train = y.loc[split['train_indices']]
    model = fitted_models[seed]
    fitted_preprocessor = model.named_steps['preprocessor']
    encoder = fitted_preprocessor.named_transformers_['categorical']
    scaler = fitted_preprocessor.named_transformers_['numeric']
    np.testing.assert_allclose(scaler.mean_, X_train[numeric_columns].mean(), rtol=1e-10, atol=1e-10)
    np.testing.assert_allclose(scaler.var_, X_train[numeric_columns].var(ddof=0), rtol=1e-10, atol=1e-10)
    assert int(scaler.n_samples_seen_) == len(X_train)
    categories = {}
    unseen = {}
    for column, learned in zip(CATEGORICAL_COLUMNS, encoder.categories_):
        assert set(learned) == set(X_train[column].unique())
        categories[column] = learned.tolist()
        unseen[column] = sorted(set(X_validation[column]) - set(learned))
    preprocessing_checks.append({
        'run': split['run'], 'seed': seed, 'numeric_columns': numeric_columns,
        'scaler_mean': scaler.mean_.tolist(), 'scaler_var': scaler.var_.tolist(),
        'scaler_n_samples_seen': int(scaler.n_samples_seen_),
        'categories': categories, 'unseen_validation': unseen,
    })
    classifier = model.named_steps['classifier']

    assert np.isfinite(classifier.cluster_centers_).all()
    assert classifier.labels_.shape == (len(X_train),)
    details = {
        'n_iter': int(classifier.n_iter_), 'inertia': float(classifier.inertia_),
        'n_init': classifier.n_init, 'n_clusters': classifier.n_clusters,
        'reached_max_iter': bool(classifier.n_iter_ >= classifier.max_iter),
        'warnings': fit_warnings_by_seed[str(seed)],
    }
    classifier_checks.append({'seed': seed, **details})
display(pd.DataFrame([{k: v for k, v in row.items() if k != 'loss_curve'} for row in classifier_checks]))
print('Encoder, scaler e classificatore verificati sui soli record di training.')

### 5b. Assegnare una famiglia ai cluster, solo sul training

Costruiamo con `pd.crosstab` la tabella cluster × famiglia dai soli record di
training. `reindex` rende visibili tutti300 cluster e tuttecinque classi. `idxmax`
sceglie la maggioranza; in parità prende la prima famiglia in FAMILY_ORDER.
Un cluster vuoto usa la maggioranza globale del training. Mostriamo numero di
cluster assegnati a ogni famiglia, pareggi e classi che non eleggono alcun cluster.


In [ ]:
training_assignments = {}
training_targets = {}
for split in prepared_splits:
    seed = split['seed']
    training_assignments[seed] = fitted_models[seed].named_steps['classifier'].labels_.copy()
    training_targets[seed] = y.loc[split['train_indices']].to_numpy()


In [ ]:
cluster_to_family = {}
cluster_tables = {}
cluster_vote_checks = []
for seed, assignments in training_assignments.items():
    targets = training_targets[seed]
    counts = pd.crosstab(pd.Series(assignments, name='cluster'), pd.Series(targets, name='family'))
    counts = counts.reindex(index=range(cluster_count), columns=FAMILY_ORDER, fill_value=0)
    global_majority = counts.sum(axis=0).idxmax()
    empty = counts.sum(axis=1).eq(0)
    tied = counts.eq(counts.max(axis=1), axis=0).sum(axis=1).gt(1) & ~empty
    mapping = counts.idxmax(axis=1)
    mapping.loc[empty] = global_majority
    cluster_to_family[seed] = mapping.to_numpy(dtype=str)
    cluster_tables[seed] = counts
    cluster_vote_checks.append({
        'seed': seed, 'counts': counts.to_numpy().tolist(), 'mapping': mapping.tolist(),
        'tied_clusters': counts.index[tied].tolist(), 'empty_clusters': counts.index[empty].tolist(),
        'global_training_majority': global_majority,
        'unrepresented_families': [family for family in FAMILY_ORDER if family not in set(mapping)],
    })
    display(mapping.value_counts().reindex(FAMILY_ORDER, fill_value=0).rename(f'cluster assegnati, seed{seed}'))
display(pd.DataFrame([{key: value for key, value in row.items() if key not in ['counts', 'mapping']}
                      for row in cluster_vote_checks]))


## 6. Predizione: dal cluster alla classe

`Pipeline.predict(X_validation)` applica il preprocessing già appreso e restituisce
il numero del cluster più vicino. Lo trasformiamo in famiglia con il mapping di
training congelato sopra. Nessun fit o voto viene eseguito sulla validation.
Salviamo anche i cluster per poter verificare questa trasformazione.


In [ ]:
prediction_arrays = {}
predict_seconds_by_seed = {}
for split in prepared_splits:
    run, seed = split['run'], split['seed']
    validation_indices = split['validation_indices']
    X_validation = X.loc[validation_indices]
    start = perf_counter()
    validation_clusters = fitted_models[seed].predict(X_validation)
    y_pred = cluster_to_family[seed][validation_clusters]
    prediction_arrays[f'run_{run:02d}_validation_clusters'] = validation_clusters
    predict_seconds_by_seed[str(seed)] = perf_counter() - start
    prediction_arrays[f'run_{run:02d}_validation_indices'] = validation_indices.copy()
    prediction_arrays[f'run_{run:02d}_truth'] = np.asarray(split['y_validation'], dtype=str)
    prediction_arrays[f'run_{run:02d}_predicted'] = np.asarray(y_pred, dtype=str)
    print(f"Predizione seed={seed}: {predict_seconds_by_seed[str(seed)]:.2f} secondi.", flush=True)

## 7. Metriche per run e per famiglia

La matrice ha **righe = classe vera, colonne = classe predetta**. Normalizzando
ciascuna riga, la diagonale è il recall: la quota dei casi reali riconosciuti.
`quota_errori = 1 - recall`. Per capire quanto fidarsi di una previsione di attacco
leggiamo anche precision e falsi positivi.

Se una classe non viene mai predetta, impostiamo precision/F1 a zero con
`zero_division=0`, mantenendo supporto ed errori visibili. Le metriche si calcolano
prima per run, poi si mediano; non ricaviamo la precision dalla matrice media.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

run_rows = []
class_rows = []
count_matrices = []
normalized_matrices = []
for run, seed in enumerate(seeds, start=1):
    truth = prediction_arrays[f'run_{run:02d}_truth']
    predicted = prediction_arrays[f'run_{run:02d}_predicted']
    report = classification_report(truth, predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    count_matrices.append(counts)
    normalized_matrices.append(confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true'))
    run_rows.append({
        'run': run, 'seed': seed, 'accuracy': accuracy_score(truth, predicted),
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
    })
    for i, family in enumerate(FAMILY_ORDER):
        support, correct = int(counts[i].sum()), int(counts[i, i])
        class_rows.append({
            'run': run, 'seed': seed, 'famiglia': family,
            'support': support, 'corretti': correct, 'errori': support - correct,
            'falsi_positivi': int(counts[:, i].sum() - correct),
            'quota_errori': (support - correct) / support if support else np.nan,
            'precision': report[family]['precision'], 'recall': report[family]['recall'],
            'f1': report[family]['f1-score'],
        })
run_results = pd.DataFrame(run_rows)
class_results = pd.DataFrame(class_rows)

In [ ]:
overall_summary = run_results[['accuracy', 'macro_f1', 'weighted_f1']].agg(['mean', 'std', 'min', 'max'])
class_summary = class_results.groupby('famiglia', sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
].agg(['mean', 'std', 'min', 'max'])
mean_matrix = np.stack(normalized_matrices).mean(axis=0)
std_matrix = np.stack(normalized_matrices).std(axis=0, ddof=1)
display(run_results.round(6))
display(overall_summary.round(6))
display(class_summary.reindex(FAMILY_ORDER).round(4))
display(class_results.loc[class_results['famiglia'].isin(['R2L', 'U2R'])].round(4))

## 8. Visualizzare la confusion matrix normalizzata

È la **media delle cinque matrici normalizzate per riga**, con scala comune 0–1.
La tabella successiva mostra la deviazione standard campionaria delle celle.
La matrice descrive errori fra classi, non correlazioni fra feature.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

figure, axis = plt.subplots(figsize=(6.6, 5.4), layout='constrained')
ConfusionMatrixDisplay(mean_matrix, display_labels=FAMILY_ORDER).plot(
    ax=axis, cmap='Blues', values_format='.2f', colorbar=True, im_kw={'vmin': 0, 'vmax': 1},
)
axis.set(title=f'{model_id}: media su 5 split', xlabel='Classe predetta', ylabel='Classe vera')
plt.show()
display(pd.DataFrame(std_matrix, index=FAMILY_ORDER, columns=FAMILY_ORDER).round(4))

## 9. Salvare risultati e provenienza

Gli artefatti di questo modello sono separati dai precedenti: CSV leggibili,
JSON con configurazione e controlli, NPZ Unicode rileggibile con `allow_pickle=False`.
Non salviamo i modelli addestrati. I tempi si riferiscono a questa macchina.
Il nuovo schema non usa `max_depth`, che non è un parametro di questi modelli.

In [ ]:
artifacts = project_root / 'artifacts' / model_id
artifacts.mkdir(parents=True, exist_ok=True)
run_results.to_csv(artifacts / 'run_metrics.csv', index=False)
class_results.to_csv(artifacts / 'class_metrics.csv', index=False)
overall_summary.to_csv(artifacts / 'overall_summary.csv')
class_summary.to_csv(artifacts / 'class_summary.csv')
np.savez_compressed(artifacts / 'predictions.npz', **prediction_arrays)
figure.savefig(artifacts / 'mean_confusion_matrix.png', dpi=150)
pd.DataFrame(mean_matrix, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(artifacts / 'confusion_mean.csv')
pd.DataFrame(std_matrix, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(artifacts / 'confusion_std.csv')
experiment = {
    'schema_version': 2, 'model_id': model_id,
    'evaluation_set': 'repeated validation from KDDTrain+ only',
    'source': reference['source'], 'source_sha256': source_sha256,
    'rows': len(X), 'feature_count': X.shape[1], 'family_order': list(FAMILY_ORDER),
    'validation_size': 0.2, 'stratified': True, 'seeds': seeds, 'std_ddof': 1,
    'randomness_varied': ['train_validation_split', 'model_initialization'], 'splits': split_records,
    'classifier_parameters_by_seed': parameters_by_seed,
    'preprocessing': {'categorical': 'OneHotEncoder(handle_unknown=ignore, sparse_output=False)',
                      'numeric': 'StandardScaler()', 'fit_scope': 'training partition only'},
    'preprocessing_checks': preprocessing_checks, 'classifier_checks': classifier_checks,
    'cluster_vote_checks': cluster_vote_checks,
    'vote_policy': {'ties': 'FAMILY_ORDER', 'empty': 'global_training_majority'},
    'fit_warnings_by_seed': fit_warnings_by_seed,
    'fit_seconds_by_seed': fit_seconds_by_seed, 'predict_seconds_by_seed': predict_seconds_by_seed,
    'run_metrics': run_results.to_dict(orient='records'),
    'class_metrics': class_results.to_dict(orient='records'),
    'normalized_confusion_mean': mean_matrix.tolist(),
    'normalized_confusion_std': std_matrix.tolist(),
    'confusion_counts': [matrix.tolist() for matrix in count_matrices],
    'unique_validation_counts': reference['unique_validation_counts'],
    'versions': {'python': platform.python_version(), **{
        package: importlib.metadata.version(package) for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']
    }},
    'limits': [
        'Overlapping repeated holdouts; measurements are not independent.',
        'Rare U2R cases are reused; repetitions do not create new observations.',
        'Known duplicate-feature overlap is retained; see DATA_AUDIT.md.',
        'One initial configuration, all 41 features; no official-test estimate or exact paper replication.',
        'Models use different preprocessing/class-weight choices explicitly documented; no isolated causal attribution.',
    ],
}
(artifacts / 'results.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
timings = pd.DataFrame({
    'seed': seeds, 'fit_seconds': [fit_seconds_by_seed[str(seed)] for seed in seeds],
    'predict_seconds': [predict_seconds_by_seed[str(seed)] for seed in seeds],
})
timings.to_csv(artifacts / 'timings.csv', index=False)
display(timings.round(3))
cluster_audit_arrays = {}
for split in prepared_splits:
    run, seed = split['run'], split['seed']
    cluster_audit_arrays[f'run_{run:02d}_train_indices'] = split['train_indices']
    cluster_audit_arrays[f'run_{run:02d}_train_clusters'] = training_assignments[seed]
    cluster_audit_arrays[f'run_{run:02d}_centers'] = fitted_models[seed].named_steps['classifier'].cluster_centers_
    cluster_tables[seed].to_csv(artifacts / f'cluster_counts_seed_{seed}.csv', index_label='cluster')
np.savez_compressed(artifacts / 'cluster_audit.npz', **cluster_audit_arrays)


## 10. Confrontare gli stessi casi con i modelli precedenti

Riutilizziamo le predizioni salvate, senza nuovi fit degli alberi. Ricontrolliamo
fonte, seed, indici e target prima di affiancare i risultati; le statistiche vengono
ricalcolate dalle predizioni. Il primo albero è senza pesi, albero pesato e RF sono
`balanced`, NB/SVM/rete/K-Means senza pesi aggiunti: confrontiamo queste configurazioni,
non attribuiamo ogni differenza al solo algoritmo. Il preprocessing è dichiarato
per ciascun metodo e appreso dal rispettivo training.

In [ ]:
saved_models = {
    'decision_tree': 'rare_class_reference',
    'decision_tree_balanced': 'decision_tree_balanced',
    'random_forest_balanced': 'random_forest_balanced',
    'gaussian_naive_bayes': 'gaussian_naive_bayes',
    'svm_rbf': 'svm_rbf',
    'neural_network': 'neural_network',
}
comparison_overall_rows = []
comparison_class_rows = []
comparison_matrices = {}
for compared_id, folder in {**saved_models, model_id: model_id}.items():
    path = project_root / 'artifacts' / folder
    metadata = json.loads((path / 'results.json').read_text(encoding='utf-8'))
    if metadata.get('schema_version', 1) != (2 if compared_id in {'gaussian_naive_bayes', 'svm_rbf', 'neural_network', 'kmeans'} else 1):
        raise ValueError(f'Formato artefatto non supportato per {compared_id}.')
    if metadata.get('model_id', 'decision_tree' if compared_id == 'decision_tree' else None) != compared_id:
        raise ValueError(f'Identità artefatto diversa dal modello atteso: {compared_id}.')
    assert metadata['source_sha256'] == source_sha256 and metadata['seeds'] == seeds
    assert metadata['feature_count'] == 41 and metadata['family_order'] == list(FAMILY_ORDER)
    assert metadata['splits'] == split_records
    matrices = []
    with np.load(path / 'predictions.npz', allow_pickle=False) as saved:
        for run, seed in enumerate(seeds, start=1):
            np.testing.assert_array_equal(saved[f'run_{run:02d}_validation_indices'],
                                          prediction_arrays[f'run_{run:02d}_validation_indices'])
            truth = saved[f'run_{run:02d}_truth']
            np.testing.assert_array_equal(truth, prediction_arrays[f'run_{run:02d}_truth'])
            suffix = 'predicted' if metadata.get('schema_version') == 2 else 'depth_15'
            predicted = saved[f'run_{run:02d}_{suffix}']
            counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
            recorded_counts = (metadata['confusion_counts'][run - 1] if metadata.get('schema_version') == 2
                               else metadata['confusion_counts_by_depth']['15'][run - 1])
            if not np.array_equal(counts, recorded_counts):
                raise ValueError(f'Conteggi JSON e predizioni NPZ discordanti per {compared_id}, run{run}')
            matrices.append(confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true'))
            report = classification_report(truth, predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0)
            comparison_overall_rows.append({
                'model': compared_id, 'seed': seed, 'accuracy': accuracy_score(truth, predicted),
                'macro_f1': report['macro avg']['f1-score'], 'weighted_f1': report['weighted avg']['f1-score'],
            })
            for i, family in enumerate(FAMILY_ORDER):
                comparison_class_rows.append({
                    'model': compared_id, 'seed': seed, 'famiglia': family,
                    'precision': report[family]['precision'], 'recall': report[family]['recall'],
                    'f1': report[family]['f1-score'], 'quota_errori': 1 - report[family]['recall'],
                    'falsi_positivi': int(counts[:, i].sum() - counts[i, i]),
                })
    comparison_matrices[compared_id] = np.stack(matrices).mean(axis=0)
comparison_overall = pd.DataFrame(comparison_overall_rows)
comparison_classes = pd.DataFrame(comparison_class_rows)
comparison_summary = comparison_overall.groupby('model', sort=False)[
    ['accuracy', 'macro_f1', 'weighted_f1']
].agg(['mean', 'std', 'min', 'max'])
comparison_class_summary = comparison_classes.groupby(['model', 'famiglia'], sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
].agg(['mean', 'std', 'min', 'max'])
display(comparison_summary.round(6))
display(comparison_class_summary.loc[(slice(None), ['R2L', 'U2R']), :].round(4))
comparison_overall.to_csv(artifacts / 'comparison_run_metrics.csv', index=False)
comparison_classes.to_csv(artifacts / 'comparison_class_metrics.csv', index=False)
comparison_summary.to_csv(artifacts / 'comparison_overall_summary.csv')
comparison_class_summary.to_csv(artifacts / 'comparison_class_summary.csv')

In [ ]:
metrics = ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
reference_metrics = comparison_classes.loc[comparison_classes['model'].eq('decision_tree')]
paired = comparison_classes.loc[comparison_classes['model'].eq(model_id)].merge(
    reference_metrics, on=['seed', 'famiglia'], suffixes=('_model', '_reference'), validate='one_to_one',
)
paired_deltas = paired[['seed', 'famiglia']].copy()
for metric in metrics:
    paired_deltas[metric] = paired[f'{metric}_model'] - paired[f'{metric}_reference']
paired_deltas.to_csv(artifacts / 'paired_deltas_vs_tree.csv', index=False)
display(paired_deltas.groupby('famiglia', sort=False)[metrics].agg(['mean', 'std', 'min', 'max']).round(4))

In [ ]:
comparison_figure, axes = plt.subplots(2, 4, figsize=(20, 9), layout='constrained')
for axis, (compared_id, matrix) in zip(axes.flat, comparison_matrices.items()):
    ConfusionMatrixDisplay(matrix, display_labels=FAMILY_ORDER).plot(
        ax=axis, cmap='Blues', values_format='.2f', colorbar=False, im_kw={'vmin': 0, 'vmax': 1},
    )
    axis.set(title=compared_id, xlabel='Classe predetta', ylabel='Classe vera')
for axis in list(axes.flat)[len(comparison_matrices):]:
    axis.set_visible(False)
comparison_figure.colorbar(axes.flat[0].images[0], ax=list(axes.flat)[:len(comparison_matrices)], shrink=0.7)
comparison_figure.suptitle('Stessi cinque split: medie delle matrici normalizzate per classe vera')
comparison_figure.savefig(artifacts / 'comparison_confusion_matrices.png', dpi=150)
plt.show()

## 11. Leggere i risultati senza nascondere le classi rare

La diagonale delle matrici normalizzate è il recall per classe; errore=1−recall.
Affiancare precision e falsi positivi: un recall alto da solo può indicare tanti
allarmi errati. Leggere anche i singoli seed e le classi mai predette.
Le cinque validation si sovrappongono: includono31 U2R distinti,10 per run.
La DS descrive dispersione, non un intervallo di confidenza o cinque dataset nuovi.
Confrontiamo configurazioni con pesi/preprocessing diversi, non solo gli algoritmi.
L'interpretazione misurata viene aggiunta dopo il controllo indipendente dei NPZ.


## K-Means300 + voto — risultato misurato e ricontrollato

Cinque fit sugli split 80/20 seed 1–5. **Macro F1 medio 0.759312 ± 0.048075 DS**, accuracy 98.10%.

| Famiglia | Precision | Recall | Errore | FP medi/run |
|---|---|---|---|---|
| R2L | 78.98% | 76.28% | 23.72% | 42.8 |
| U2R | 11.43% | 8.00% | 92.00% | 1.6 |

Famiglie che non eleggono alcun cluster, per seed: `{1: [], 2: [], 3: [], 4: [], 5: []}`. Se una famiglia non vince alcun voto non può essere predetta: precision e recall restano a zero, senza eliminarla dalla matrice o dal macro F1.

In queste prove tutte le famiglie eleggono almeno un cluster nel training. I casi U2R corretti nei seed 1–5 sono [0, 4, 0, 0, 0]: quattro split non ne riconoscono nessuno. Il limite osservato non deriva dalla totale assenza di un cluster U2R nel training; gli assegnamenti non riconoscono bene gli U2R lasciati in validation.

Il voto è stato ricalcolato dalle sole etichette training e i cluster di validation ricontrollati dai centroidi salvati. Non ri-etichettiamo i cluster per migliorare la matrice.

| Seed | Iterazioni del restart scelto | Limite raggiunto | Warning | Fit: secondi |
|---|---|---|---|---|
| 1 | 46 | False | 0 | 59.46 |
| 2 | 36 | False | 0 | 58.45 |
| 3 | 54 | False | 0 | 60.23 |
| 4 | 29 | False | 0 | 57.77 |
| 5 | 44 | False | 0 | 62.98 |

Metriche ricalcolate indipendentemente dai NPZ. Stessi indici e target dei riferimenti; le validation si sovrappongono e riutilizzano 31 U2R distinti. Il confronto non è un test ufficiale o una replica numerica del paper.

### Controllo conclusivo del confronto

I conteggi ricalcolati da ogni NPZ devono coincidere con quelli registrati nel
JSON del modello. Il controllo intercetta anche predizioni incompatibili che
condividono indici e target. Correzione verificata con nove regressioni e confronto
reale rieseguito senza fit; nessun risultato misurato cambiato. Suite completa:67 test.
